# Phase 2D-B | DANDI 000005: Multi-animal optogenetics feasibility audit

**Research project:** Optogenetic Neural Digital Twin (data feasibility, **not** model training)  
**Dandiset:** [000005](https://dandiarchive.org/dandiset/000005/0.220126.1853) | Published version `0.220126.1853`  
**Source paper:** [Yu et al., Nature Neuroscience (2016)](https://doi.org/10.1038/nn.4412)

**Key question:** Do multiple independent mice have **recorded neural activity AND usable light stimulation timing in the same NWB file**, on an interpretable time axis?

**Important limitations:** This dataset concerns mouse somatosensory/thalamocortical activity, **not human hippocampal epilepsy**. Intracellular voltage (`CurrentClampSeries`) may exist *without* sorted spike times. We inspect both types. An `+ogen` filename or `OptogeneticSeries` entry does **not** guarantee explicit light ON/OFF events, and timestamp overlap alone does **not** prove synchronized clocks or causation.

## 1. Kaggle instructions and safe file limits

1. Upload this notebook to Kaggle.
2. Enable **Internet ON**. Select **Accelerator: None** (GPU not needed).
3. Choose **Run All**. No additional Kaggle Input is needed.
4. Download **`phase2db_dandi000005_audit_results.zip`** from `/kaggle/working` and upload it to the research chat.

This first run downloads **at most 3 different mouse files** (targeting about 183 MB altogether). Size caps are enforced in the downloader. Full Dandiset size is about 46.4 GB; **do not download the whole Dandiset**.

The report distinguishes: **manifest-level candidates**, **actual inspected neural signals**, **actual optical series**, **candidate light ON/OFF intervals**, and **provisional time overlap**. No synthetic data are included in the Kaggle outputs.

In [1]:
import os, sys, json, re, zipfile, traceback, time
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import requests
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
from IPython.display import display
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

DANDI_ID = '000005'
VERSION = '0.220126.1853'  # Published reproducible version
API = 'https://api.dandiarchive.org'
OUT = Path('/kaggle/working/phase2db_dandi000005_audit') if Path('/kaggle/working').exists() else Path('/mnt/data/phase2db_local')
CACHE = OUT / '_temporary_downloads'
OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

MAX_SUBJECTS = 3
MAX_ASSET_MB = 85.0
MAX_TOTAL_MB = 205.0
MAX_HDF5_NODES = 30000
MAX_OPTICAL_POINTS_FULL_SCAN = 2_000_000
MAX_EXAMPLE_EVENTS = 1000
MAX_ANALOG_POINTS_FOR_PREVIEW = 3000
REMOVE_TEMP_NWB_AFTER_SCAN = True

HTTP = requests.Session()
HTTP.mount('https://', HTTPAdapter(max_retries=Retry(
    total=3, connect=3, read=2, backoff_factor=1.0,
    status_forcelist=[429, 500, 502, 503, 504], allowed_methods=['GET'])))
HTTP.headers.update({'User-Agent':'DANDI-optogenetic-feasibility-audit/1.0'})
ERRORS = []
print('Dataset:', DANDI_ID, VERSION)
print('CPU-only metadata/recording audit. Output:', OUT)
print('Size limits: <=',MAX_ASSET_MB,'MB per asset; <=',MAX_TOTAL_MB,'MB total')

Dataset: 000005 0.220126.1853
CPU-only metadata/recording audit. Output: /kaggle/working/phase2db_dandi000005_audit
Size limits: <= 85.0 MB per asset; <= 205.0 MB total


## 2. Manifest audit: actual file counts and candidate mice

The archive metadata lists **148 files / 55 subject folders**. The notebook recomputes these numbers from the pinned published manifest. We select optogenetics-marked files from **three distinct subject folders**, sorted by download size. A subject folder is a *candidate animal identifier*; we verify it against the NWB subject metadata once the file opens.

In [2]:
def api_json(url):
    response = HTTP.get(url, timeout=(20,90), headers={'Accept':'application/json'})
    response.raise_for_status()
    payload = response.json()
    if not isinstance(payload, dict):
        raise ValueError('Unexpected API JSON payload')
    return payload

def get_manifest():
    url = f'{API}/api/dandisets/{DANDI_ID}/versions/{VERSION}/assets/?page_size=1000'
    entries = []
    for _ in range(100):
        body = api_json(url)
        entries.extend(body.get('results', []))
        url = body.get('next')
        if not url:
            break
    else:
        raise RuntimeError('Manifest pagination exceeded 100 pages')
    if not entries:
        raise RuntimeError('No assets available. Check Internet ON and version.')
    data = pd.DataFrame(entries)
    for col in ('path','size','asset_id'):
        if col not in data:
            raise KeyError('Missing DANDI asset field '+col)
    data['size_bytes'] = pd.to_numeric(data['size'], errors='coerce')
    data['size_MB'] = data['size_bytes'] / 1e6
    data['subject_folder'] = data['path'].str.extract(r'(sub-[^/]+)', expand=False)
    data['is_nwb'] = data['path'].str.endswith('.nwb', na=False)
    data['opto_filename'] = data['path'].str.contains(r'\+ogen\.nwb$', regex=True, na=False)
    data['intracellular_filename'] = data['path'].str.contains(r'\+icephys', regex=True, na=False)
    return data

manifest = get_manifest()
manifest.to_csv(OUT/'full_published_manifest_000005.csv', index=False)
marked = manifest.loc[manifest.is_nwb & manifest.opto_filename].copy().sort_values('size_bytes')
smallest_by_subject = marked.drop_duplicates('subject_folder').sort_values('size_bytes').reset_index(drop=True)
eligible = smallest_by_subject.loc[smallest_by_subject.size_MB.le(MAX_ASSET_MB)].copy()
selected = eligible.head(MAX_SUBJECTS).copy()
selected.to_csv(OUT/'selected_three_mice.csv', index=False)
manifest_summary = {
    'dandiset_id':DANDI_ID,'version':VERSION,
    'total_assets':int(len(manifest)),
    'subject_folders_all':int(manifest.subject_folder.nunique()),
    'opto_marked_files':int(len(marked)),
    'opto_marked_subject_folders':int(marked.subject_folder.nunique()),
    'opto_marked_icephys_files':int((marked.intracellular_filename).sum()),
    'minimum_three_distinct_mouse_files_MB':float(smallest_by_subject.head(3).size_MB.sum()),
    'archive_total_GB':float(manifest.size_bytes.sum()/1e9),
    'note':'These are filename-based metadata counts only.'
}
(OUT/'manifest_summary.json').write_text(json.dumps(manifest_summary, indent=2))
print(json.dumps(manifest_summary, indent=2))
if len(selected)<3:
    print('WARNING: fewer than 3 candidate subjects meet size cap; results may be inconclusive.')
print('First-run selected files (three distinct subject folders):')
display(selected[['subject_folder','size_MB','path']])

{
  "dandiset_id": "000005",
  "version": "0.220126.1853",
  "total_assets": 148,
  "subject_folders_all": 55,
  "opto_marked_files": 121,
  "opto_marked_subject_folders": 43,
  "opto_marked_icephys_files": 121,
  "minimum_three_distinct_mouse_files_MB": 183.056192,
  "archive_total_GB": 46.436686324,
  "note": "These are filename-based metadata counts only."
}
First-run selected files (three distinct subject folders):


,subject_folder,size_MB,path
0,sub-anm266945,46.543184,sub-anm266945/sub-anm266945_ses-20141221_obj-1...
1,sub-anm266951,58.123096,sub-anm266951/sub-anm266951_ses-20141201_behav...
2,sub-anm281592,78.389912,sub-anm281592/sub-anm281592_ses-20150309_behav...


## 3. Bounded download of three small files

Only the selected files are downloaded. A redirect to the public DANDI object store is expected. Transfers abort if the published size is exceeded. The notebook will still export an error report if an individual file cannot be fetched. Large voltage waveforms are **not loaded into RAM** during the inspection.

In [3]:
def fetch_selected_file(row):
    expected = int(row['size_bytes'])
    if not (int(expected)>0 and int(expected)<=MAX_ASSET_MB*1e6):
        raise ValueError('File not within permitted size cap')
    if not bool(row['opto_filename']):
        raise ValueError('Refusing file without +ogen filename marker')
    dest = CACHE / (str(row['asset_id'])+'.nwb')
    if dest.exists() and dest.stat().st_size == expected:
        return dest,0
    part = CACHE / (str(row['asset_id'])+'.part')
    received=0
    try:
        with HTTP.get(f"{API}/api/assets/{row['asset_id']}/download/",
                      stream=True, headers={'Accept':'*/*'}, timeout=(30,120)) as response:
            response.raise_for_status()
            with open(part, 'wb') as handle:
                for block in response.iter_content(chunk_size=1024*512):
                    if not block: continue
                    received += len(block)
                    if received>expected+4096 or received>MAX_ASSET_MB*1e6:
                        raise RuntimeError('Downloaded more bytes than allowed')
                    handle.write(block)
        if received!=expected:
            raise IOError(f'File size mismatch expected {expected}, received {received}')
        part.replace(dest)
        return dest,received
    finally:
        if part.exists(): part.unlink()

files={}
used_bytes=0
for _,row in selected.iterrows():
    if used_bytes + int(row['size_bytes']) > MAX_TOTAL_MB*1e6:
        ERRORS.append({'stage':'budget','subject':row['subject_folder'],'file':row['path'],
                       'error':'Skipped: max total download cap reached'})
        continue
    try:
        path, transfer = fetch_selected_file(row)
        files[str(row['asset_id'])] = path
        used_bytes += transfer
        print('OK:',row['subject_folder'],round(path.stat().st_size/1e6,2),'MB')
    except Exception as exc:
        ERRORS.append({'stage':'download','subject':row['subject_folder'],'file':row['path'],'error':repr(exc)})
        print('Download failed for',row['subject_folder'],repr(exc))
print('Files available:',len(files),'Newly transferred MB:',round(used_bytes/1e6,2))

OK: sub-anm266945 46.54 MB
OK: sub-anm266951 58.12 MB
OK: sub-anm281592 78.39 MB
Files available: 3 Newly transferred MB: 183.06


## 4. Extract optical series, neural activity, and timestamp information

We inspect HDF5/NWB structures directly using `h5py`:

- **Optical:** `OptogeneticSeries` groups and light/laser time series, data type, units, timestamp method, active/zero transitions (only if justified by raw values).
- **Neural:** sorted `units/spike_times`, or acquired `CurrentClampSeries`, `VoltageClampSeries`, or `ElectricalSeries` (intracellular voltage or extracellular recording).
- **Time:** start/end of each type of recording. Check temporal intersection, **not assumed clock synchronization**.
- **Caution:** `CurrentClampStimulusSeries` is electrical current injected through a pipette, **not neural response and not optical intervention**.

If an optical series has only constant values, unknown encoding, or missing timestamps, mark the trial count **unverified** instead of inventing light events.

In [4]:
def txt(value):
    if isinstance(value, bytes):
        return value.decode('utf8','replace')
    if isinstance(value, np.ndarray):
        if value.ndim==0: return txt(value[()])
        if value.size<=1: return txt(value.reshape(-1)[0]) if value.size else ''
        return '|'.join(txt(x) for x in value.reshape(-1)[:6])
    return str(value)

def safe_scalar(dataset, fallback=np.nan):
    if not isinstance(dataset, h5py.Dataset): return fallback
    try:
        val = dataset[()]
        return float(np.asarray(val).reshape(-1)[0])
    except Exception:
        return fallback

def time_info(group):
    data = group.get('data')
    if not isinstance(data,h5py.Dataset) or data.ndim==0:
        return (np.nan,np.nan,'no_signal_array',0)
    n = int(data.shape[0])
    timestamps = group.get('timestamps')
    if isinstance(timestamps,h5py.Dataset) and timestamps.ndim==1 and len(timestamps):
        try:
            return float(timestamps[0]),float(timestamps[-1]),'explicit_timestamps',n
        except Exception: pass
    start = group.get('starting_time')
    if isinstance(start,h5py.Dataset):
        t0=safe_scalar(start)
        try: rate = float(np.asarray(start.attrs.get('rate', np.nan)).reshape(-1)[0])
        except Exception: rate=np.nan
        if np.isfinite(t0) and np.isfinite(rate) and rate>0:
            return t0,t0+(n-1)/rate,'starting_time_rate',n
    return (np.nan,np.nan,'missing_time_axis',n)

def group_neurotype(g):
    return txt(g.attrs.get('neurodata_type','')).split('|')[0]

def group_paths(hf):
    result=[]
    visited=[0]
    def visitor(path,obj):
        visited[0]+=1
        if visited[0]>MAX_HDF5_NODES:
            return True
        if not isinstance(obj,h5py.Group): return
        name=path.lower(); t=group_neurotype(obj).lower()
        if isinstance(obj.get('data'), h5py.Dataset):
            opt = ('optogeneticseries' in t or
                   (any(k in name for k in ['opto','optogen','laser','photo','light']) and
                    not any(k in t for k in ['stimulationsite'])))
            neuron = (path.startswith('acquisition/') and
                     (any(k in t for k in ['currentclampseries','voltageclampseries','electricalseries','spikeeventseries'])
                      or any(k in name for k in ['voltage','membrane','intracellular','electrical'])))
            if opt or neuron:
                result.append((path, 'optical' if opt else 'neural', t))
    hf.visititems(visitor)
    return result,visited[0]

def inspect_units(hf):
    candidates=[]
    if isinstance(hf.get('/units'),h5py.Group): candidates.append(('/units',hf['/units']))
    # Secondary formats with processed unit table.
    paths,_=group_paths(hf)
    for path,kind,t in paths: pass
    for path,g in candidates:
        spikes = g.get('spike_times')
        if not isinstance(spikes,h5py.Dataset) or spikes.ndim!=1: continue
        nspikes=int(spikes.shape[0])
        idx=g.get('spike_times_index')
        ids=g.get('id')
        units=int(len(ids)) if isinstance(ids,h5py.Dataset) else (int(len(idx)) if isinstance(idx,h5py.Dataset) else 0)
        mn,mx=np.nan,np.nan
        if nspikes:
            # Bounded sampling; spike-times per-unit need not be globally time-sorted.
            step=max(1, nspikes//5000)
            sampled=np.asarray(spikes[::step],dtype=float)
            sampled=sampled[np.isfinite(sampled)]
            if sampled.size: mn,mx=float(sampled.min()),float(sampled.max())
        return {'n_sorted_units':units,'n_spikes':nspikes,'spike_start_s_approx':mn,
                'spike_end_s_approx':mx,'unit_time_source':'subsampled spike-times, approximate bounds'}
    return {'n_sorted_units':0,'n_spikes':0,'spike_start_s_approx':np.nan,
            'spike_end_s_approx':np.nan,'unit_time_source':'No compatible /units table'}

def sample_analog_preview(g):
    data=g.get('data')
    if not isinstance(data,h5py.Dataset) or data.ndim==0: return []
    k=min(int(data.shape[0]), MAX_ANALOG_POINTS_FOR_PREVIEW)
    if k<=0:return []
    try:
        if data.ndim==1:
            values=np.asarray(data[:k],dtype=float)
        else:
            values=np.asarray(data[:k,0],dtype=float)
        return values[np.isfinite(values)].tolist()[:1000]
    except Exception:
        return []

def inspect_optical(hf,gpath,meta):
    g=hf[gpath]
    data=g['data']
    t0,t1,source,n=time_info(g)
    unit=txt(data.attrs.get('unit',g.attrs.get('unit','unknown')))
    info={**meta, 'series_path':gpath,'series_neurodata_type':group_neurotype(g),
          'data_dtype':str(data.dtype), 'data_shape':str(tuple(data.shape)),
          'stored_unit':unit,'t_start_s':t0,'t_end_s':t1,'time_axis_method':source,
          'scanned_all_values':False,'unique_codes_preview':'','complete_on_off_pairs':0,
          'classification':'optical_timeseries_present; pulse meaning unverified'}
    events=[]
    if data.ndim!=1 or not np.issubdtype(data.dtype,np.number):
        info['classification']='unsupported optical data shape or type; no event reconstruction'
        return info,events
    if n>MAX_OPTICAL_POINTS_FULL_SCAN:
        info['classification']='optical array too large for bounded full scan; event count unknown'
        return info,events
    if n<2 or not np.isfinite(t0) or not np.isfinite(t1):
        info['classification']='missing usable time axis or fewer than two points'
        return info,events
    vals=np.asarray(data[:],dtype=np.float64)
    if not np.all(np.isfinite(vals)):
        info['classification']='nonfinite optical values; manual interpretation required'
        return info,events
    uniq=np.unique(vals)
    info['scanned_all_values']=True
    info['unique_codes_preview']='|'.join(f'{float(x):.7g}' for x in uniq[:20])
    if len(uniq)>100 or not (np.any(vals==0) and np.any(vals>0)):
        info['classification']='not a clear positive-versus-zero ON/OFF series'
        return info,events
    active=vals>0
    rises=np.flatnonzero((~active[:-1]) & active[1:])+1
    falls=np.flatnonzero(active[:-1] & (~active[1:]))+1
    if not len(rises) or not len(falls):
        info['classification']='no complete zero-positive-zero transitions'
        return info,events
    timestamps=g.get('timestamps')
    # All event positions must be inferred from the SAME optical time axis.
    if source=='starting_time_rate':
        rate=float(np.asarray(g['starting_time'].attrs['rate']).reshape(-1)[0])
        times=lambda i: t0 + i/rate
    elif isinstance(timestamps,h5py.Dataset) and len(timestamps)==n:
        times=lambda i: float(timestamps[int(i)])
    else:
        info['classification']='cannot match numeric optical values with timestamps'
        return info,events
    paired_count=0
    for a in rises:
        j=np.searchsorted(falls,a,side='right')
        if j>=len(falls):continue
        b=int(falls[j]); a=int(a)
        if b<=a:continue
        onset,offset=times(a),times(b)
        if not(np.isfinite(onset) and np.isfinite(offset) and offset>onset):continue
        paired_count+=1
        if len(events)<MAX_EXAMPLE_EVENTS:
            events.append({**meta,'series_path':gpath,'onset_sec':float(onset),
                           'offset_sec':float(offset),'duration_ms':float(1000*(offset-onset)),
                           'positive_raw_code':float(vals[a]),'stored_unit':unit})
    info['complete_on_off_pairs']=int(paired_count)
    info['classification']='candidate positive-to-zero light intervals; verify optical encoding'
    if paired_count>MAX_EXAMPLE_EVENTS:
        info['classification']+=' (CSV capped to first 1000 intervals)'
    return info,events

def overlaps(a0,a1,b0,b1):
    if not all(np.isfinite(z) for z in [a0,a1,b0,b1]):return False
    return max(a0,b0)<min(a1,b1)

def norm_subject(s):
    return str(s).strip().lower().removeprefix('sub-')

def inspect_nwb(file_path, asset_row):
    meta={'subject_folder':str(asset_row['subject_folder']),'asset_id':str(asset_row['asset_id']),
          'file_path':str(asset_row['path'])}
    row={**meta,'nwb_subject':'','subject_consistency':'unverified',
         'n_optical_timeseries':0,'n_timed_neural_acquisition_series':0,'n_sorted_units':0,'n_spikes':0,
         'n_candidate_complete_light_intervals':0,'optical_neural_time_overlap':False,
         'has_neural_signal':False,'has_time_resolved_optical_candidates':False,
         'provisionally_paired':False,'caution':'Time overlap alone is NOT clock synchronization'}
    series_out,events_out=[],[]
    with h5py.File(file_path,'r') as hf:
        sid=hf.get('/general/subject/subject_id')
        if isinstance(sid,h5py.Dataset):
            subject=txt(sid[()])
            row['nwb_subject']=subject
            row['subject_consistency']='match' if norm_subject(subject)==norm_subject(meta['subject_folder']) else 'MISMATCH'
        units=inspect_units(hf)
        row.update({k:units[k] for k in ['n_sorted_units','n_spikes']})
        paths,visited=group_paths(hf)
        row['hdf5_nodes_visited']=visited
        neural_extents=[]
        for gpath,kind,typ in paths:
            g=hf[gpath]
            if kind=='neural':
                t0,t1,method,n=time_info(g)
                entry={**meta,'series_path':gpath,'kind':'neural','neurodata_type':group_neurotype(g),
                       'data_shape':str(tuple(g['data'].shape)),
                       'unit':txt(g['data'].attrs.get('unit','unknown')),
                       't_start_s':t0,'t_end_s':t1,'time_axis_method':method,
                       'candidate_complete_events':0,'event_interpretation':''}
                series_out.append(entry)
                if np.isfinite(t0) and np.isfinite(t1) and t1>t0:
                    neural_extents.append((t0,t1,gpath))
                    row['n_timed_neural_acquisition_series']+=1
            else:
                info,events=inspect_optical(hf,gpath,meta)
                row['n_optical_timeseries']+=1
                series_out.append({**meta,'series_path':gpath,'kind':'optical',
                  'neurodata_type':info['series_neurodata_type'],
                  'data_shape':info['data_shape'],'unit':info['stored_unit'],
                  't_start_s':info['t_start_s'],'t_end_s':info['t_end_s'],
                  'time_axis_method':info['time_axis_method'],
                  'candidate_complete_events':info['complete_on_off_pairs'],
                  'event_interpretation':info['classification']})
                row['n_candidate_complete_light_intervals']+=int(info['complete_on_off_pairs'])
                events_out.extend(events)
                if info['complete_on_off_pairs']>0:
                    row['has_time_resolved_optical_candidates']=True
        row['has_neural_signal']=(row['n_sorted_units']>0 and row['n_spikes']>0) or len(neural_extents)>0
        if not neural_extents and units['n_spikes']>0:
            a,b=units['spike_start_s_approx'],units['spike_end_s_approx']
            if np.isfinite(a) and np.isfinite(b) and b>a:
                neural_extents.append((a,b,'approx_spike_extent'))
        for item in series_out:
            if item['kind']!='optical' or not item['candidate_complete_events']:continue
            if any(overlaps(item['t_start_s'],item['t_end_s'],a,b) for a,b,_ in neural_extents):
                row['optical_neural_time_overlap']=True
        row['provisionally_paired']=(row['subject_consistency']=='match' and row['has_neural_signal']
                                    and row['has_time_resolved_optical_candidates']
                                    and row['optical_neural_time_overlap'])
    return row,series_out,events_out

## 5. Execute real-file inspection (no DL training)

This cell inspects exactly the NWB files downloaded above. It saves every warning and error. A file without `/units` is **not automatically invalid**: acquired intracellular voltage with a time axis also counts as real neural activity. A file with only current injection metadata is not enough.

In [5]:
audit_rows=[]
series_rows=[]
example_events=[]
for _,item in selected.iterrows():
    file=files.get(str(item['asset_id']))
    if file is None:continue
    try:
        result,series,events=inspect_nwb(file,item)
        audit_rows.append(result)
        series_rows.extend(series)
        example_events.extend(events)
        print(item['subject_folder'], 'neural=',result['has_neural_signal'],
              'opto_series=',result['n_optical_timeseries'],
              'candidate_events=',result['n_candidate_complete_light_intervals'],
              'time_overlap=',result['optical_neural_time_overlap'],
              'provisional_pair=',result['provisionally_paired'])
    except Exception as exc:
        ERRORS.append({'stage':'inspection','subject':item['subject_folder'],'file':item['path'],
                       'error':repr(exc)+' | '+traceback.format_exc(limit=3)})
        print('Inspection issue:',item['subject_folder'],repr(exc))
    finally:
        if REMOVE_TEMP_NWB_AFTER_SCAN and file.exists():
            file.unlink()

asset_audit=pd.DataFrame(audit_rows)
series_inventory=pd.DataFrame(series_rows)
event_examples=pd.DataFrame(example_events)
asset_audit.to_csv(OUT/'actual_nwb_paired_signal_audit.csv',index=False)
series_inventory.to_csv(OUT/'optical_and_neural_series_inventory.csv',index=False)
event_examples.to_csv(OUT/'candidate_light_intervals_UNVERIFIED.csv',index=False)
pd.DataFrame(ERRORS,columns=['stage','subject','file','error']).to_csv(OUT/'errors_and_warnings.csv',index=False)
print('Files inspected',len(asset_audit),'Optical series found',
      len(series_inventory[series_inventory.kind=='optical']) if not series_inventory.empty else 0,
      'candidate complete optical transitions',int(asset_audit.n_candidate_complete_light_intervals.sum()) if not asset_audit.empty else 0)
if not asset_audit.empty:
    display(asset_audit[['subject_folder','nwb_subject','subject_consistency','n_sorted_units','n_spikes',
      'n_timed_neural_acquisition_series','n_optical_timeseries',
      'n_candidate_complete_light_intervals','optical_neural_time_overlap','provisionally_paired']])

sub-anm266945 neural= True opto_series= 1 candidate_events= 5 time_overlap= True provisional_pair= True
sub-anm266951 neural= True opto_series= 1 candidate_events= 5 time_overlap= True provisional_pair= True
sub-anm281592 neural= True opto_series= 1 candidate_events= 298 time_overlap= True provisional_pair= True
Files inspected 3 Optical series found 3 candidate complete optical transitions 308


,subject_folder,nwb_subject,subject_consistency,n_sorted_units,n_spikes,n_timed_neural_acquisition_series,n_optical_timeseries,n_candidate_complete_light_intervals,optical_neural_time_overlap,provisionally_paired
0,sub-anm266945,anm266945,match,0,0,1,1,5,True,True
1,sub-anm266951,anm266951,match,0,0,1,1,5,True,True
2,sub-anm281592,anm281592,match,0,0,1,1,298,True,True


## 6. Independent-animal report and grounded figures

**Do not equate:** 55 subjects listed in DANDI metadata = 55 independently validated pairs. Only files that actually contain measurable neural signals and event-like optical series can qualify as **provisional pairs**, and even these require manual protocol/timebase confirmation before ML.

In [6]:
n_inspected=len(asset_audit)
n_valid_subjects=int(asset_audit.loc[asset_audit.provisionally_paired,'subject_folder'].nunique()) if n_inspected else 0
verdict={
    'dataset_id':DANDI_ID,'published_version':VERSION,
    'generated_utc':datetime.now(timezone.utc).isoformat(),
    'manifest_subject_folders':manifest_summary['subject_folders_all'],
    'manifest_opto_marked_subject_folders':manifest_summary['opto_marked_subject_folders'],
    'files_inspected':n_inspected,
    'independent_subjects_inspected':int(asset_audit.subject_folder.nunique()) if n_inspected else 0,
    'files_with_real_neural_signal':int(asset_audit.has_neural_signal.sum()) if n_inspected else 0,
    'files_with_opto_series':int((asset_audit.n_optical_timeseries>0).sum()) if n_inspected else 0,
    'files_with_complete_candidate_light_intervals':int((asset_audit.n_candidate_complete_light_intervals>0).sum()) if n_inspected else 0,
    'provisionally_paired_independent_subjects':n_valid_subjects,
    'verified_synchronized_light_and_neural_subjects':0,
    'training_ready':False,
    'status':('PROMISING: examine protocol and additional independent animals' if n_valid_subjects>=2
              else 'INSUFFICIENT PROVISIONAL PAIRED DATA: inspect alternative files or datasets'),
    'caution':('Positive-to-zero intervals are only candidate optical events; '+
               'time-range overlap is insufficient to prove matching clocks, '+
               'stimulation effects, or calibrated intensity.')}
(OUT/'phase2db_feasibility_summary.json').write_text(json.dumps(verdict,indent=2))

# Figure 1: grounded manifest file-size candidates (no artificially generated results).
fig,ax=plt.subplots(figsize=(8,4.5))
s=smallest_by_subject.head(12).copy()
ax.barh(range(len(s)),s.size_MB.to_numpy())
ax.set_yticks(range(len(s)))
ax.set_yticklabels(s.subject_folder.str.replace('sub-','',regex=False),fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Smallest opto-marked NWB per subject (MB)')
ax.set_title('DANDI 000005 - size-ranked optogenetic file candidates')
fig.tight_layout()
fig.savefig(OUT/'fig1_size_ranked_optogenetic_candidates.png',dpi=180)
plt.close(fig)

# Figure 2: counts actually measured by the real audit.
fig,ax=plt.subplots(figsize=(8,4))
labels=['Inspected files','Neural data','Optical series','Candidate ON/OFF intervals','Provisional paired']
vals=[n_inspected,verdict['files_with_real_neural_signal'],verdict['files_with_opto_series'],
      verdict['files_with_complete_candidate_light_intervals'],n_valid_subjects]
ax.barh(labels,vals)
ax.set_xlim(0,max(3,max(vals)+0.5))
ax.set_xlabel('Files (last row: independent subject folders)')
ax.set_title('Observed DANDI 000005 audit coverage')
for i,v in enumerate(vals):ax.text(v+0.05,i,str(v),va='center')
fig.tight_layout()
fig.savefig(OUT/'fig2_actual_neural_optical_coverage.png',dpi=180)
plt.close(fig)

print(json.dumps(verdict,indent=2))

{
  "dataset_id": "000005",
  "published_version": "0.220126.1853",
  "generated_utc": "2026-10-12T08:04:21.652872+00:00",
  "manifest_subject_folders": 55,
  "manifest_opto_marked_subject_folders": 43,
  "files_inspected": 3,
  "independent_subjects_inspected": 3,
  "files_with_real_neural_signal": 3,
  "files_with_opto_series": 3,
  "files_with_complete_candidate_light_intervals": 3,
  "provisionally_paired_independent_subjects": 3,
  "verified_synchronized_light_and_neural_subjects": 0,
  "training_ready": false,
  "status": "PROMISING: examine protocol and additional independent animals",
  "caution": "Positive-to-zero intervals are only candidate optical events; time-range overlap is insufficient to prove matching clocks, stimulation effects, or calibrated intensity."
}


## 7. Package results for research review

The ZIP includes the manifest, exactly which three files were inspected, series metadata, candidate intervals (if present), subject-consistency checks, true errors, and figures. It **does not include full NWB files or unverified treatment claims**.

In [7]:
lines=[
    '# Phase 2D-B: DANDI 000005 audit',
    '',
    f"Inspected {verdict['files_inspected']} NWB files from {verdict['independent_subjects_inspected']} subject folders.",
    f"Actual neural signal files: {verdict['files_with_real_neural_signal']}.",
    f"Files containing optical time series: {verdict['files_with_opto_series']}.",
    f"Files with candidate complete ON/OFF intervals: {verdict['files_with_complete_candidate_light_intervals']}.",
    f"Independent subjects with provisional optical/neural temporal overlap: {verdict['provisionally_paired_independent_subjects']}.",
    f"Decision: {verdict['status']}.",
    '',
    'Important: Optical time-range overlap alone is not evidence of shared clock synchronization.',
    'Positive versus zero optical codes are candidate transitions and still need physical interpretation.',
    'Intracellular current-clamp voltage is not equivalent to sorted neural spikes.',
    'Somatosensory mouse data cannot validate human hippocampal epilepsy treatment.',
    'No deep learning was trained. No fabricated stimulation events were created.',
    '',
    'Source: https://dandiarchive.org/dandiset/000005/0.220126.1853',
    'Original paper: https://doi.org/10.1038/nn.4412',
]
(OUT/'READ_ME_FIRST.txt').write_text('\n'.join(lines))
name=Path('/kaggle/working/phase2db_dandi000005_audit_results.zip') if Path('/kaggle/working').exists() else Path('/mnt/data/phase2db_dandi000005_audit_results.zip')
with zipfile.ZipFile(name,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=7) as z:
    for file in sorted(OUT.iterdir()):
        if file.is_file() and file.suffix.lower() in ('.csv','.json','.png','.txt'):
            z.write(file,arcname=file.name)
print('DOWNLOAD THIS ZIP:',name,'(',round(name.stat().st_size/1e6,2),'MB )')
print('Upload this ZIP to ChatGPT. Full 46.4 GB dataset is not required.')

DOWNLOAD THIS ZIP: /kaggle/working/phase2db_dandi000005_audit_results.zip ( 0.12 MB )
Upload this ZIP to ChatGPT. Full 46.4 GB dataset is not required.
